# Milestone 1: Retrieval & Evaluation

This notebook covers:
1. Building and testing BM25, semantic, and hybrid retrievers
2. Qualitative evaluation across a query set

In [1]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_corpus # noqa: E402
from src.bm25 import BM25Retriever # noqa: E402
from src.semantic import SemanticRetriever # noqa: E402
from src.hybrid import HybridRetriever # noqa: E402

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
INDICES_DIR = PROJECT_ROOT / "indices"

# Load corpus
corpus = load_corpus(str(DATA_PROCESSED / "product_corpus.parquet"))
print(f"Corpus loaded: {len(corpus):,} products")

c:\Users\amato\miniforge3\envs\575-project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Corpus loaded: 112,590 products


## 1. BM25 Retriever

Building a keyword-based retrieval system using BM25 (Okapi BM25).

In [2]:
bm25 = BM25Retriever()

bm25_path = INDICES_DIR / "bm25_index.pkl"
if bm25_path.exists():
    bm25.load(str(bm25_path))
    print("BM25 index loaded from disk.")
else:
    bm25.build_index(corpus)
    bm25.save(str(bm25_path))
    print("BM25 index built and saved.")

results = bm25.search("vitamin C serum", top_k=5)
print("Test query: 'vitamin C serum'")
for i, r in enumerate(results):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

BM25 index loaded from disk.
Test query: 'vitamin C serum'
1. Organic Vitamin C Serum for Face-Professional Strength-Organic Vitamin C Serum-Hyaluronic Acid-Vitamin E-Naturally Derived 20% Vitamin C (2 Ounce) (score: 21.6515)
2. PREMIUM Vitamin C Serum For Face and Eyes with Hyaluronic Acid Serum - Anti Ageing & Anti Wrinkle Serum - This Vitamin C Serum Will Plump, Hydrate & Brighten, Anti Wrinkle, Anti Aging, Fades Age Spot (score: 21.6131)
3. 24K Vitamin C Serum for Face 2 PACK, Facial Serum with Vitamin C and Hyaluronic Acid, Facial Moisturizer 2 Pack Vitamin C Serum (score: 21.4078)
4. Springs Organic Vitamin C Serum For Your Face - Vitamin C Serum|Organic Vitamin C + Vitamin E + Aloe Vera + Hyaluronic Acid Serum- Effective Strength 20% Vitamin C Serum with Vegan Hyaluronic Acid Leaves Your Skin Radiant & More Beautiful By Neutralizing Free Radicals. This Anti Aging Antiwrinkles Serum Will Give You The Results in few days or we refund You With No-Question-Asked. Your! (score: 21.35

**Observations on BM25 results for "vitamin C serum":**

- BM25 performs well on this query because the search terms appear verbatim in product titles (hence "keyword retrieval").
- The top two results score highest (~21.65 and ~21.61) because they repeat "vitamin C serum" multiple times in long titles (BM25 rewards this from term frequency).
- Results 3-5 score progressively lower (~21.41 -> 21.36 -> 21.27), with each containing the phrase fewer times or in shorter titles. The scores are close but not identical which shows the subtle differences in term frequency and document length normalization.
- A limitation is visible here which is that BM25 has no understanding of *meaning*. A query like "face brightening serum with antioxidants" would miss these same products even though vitamin C serums are exactly that. This is the gap we expect semantic retrieval to fill.

## 2. Semantic Retriever

Building a semantic search system using sentence-transformers (all-MiniLM-L6-v2) and FAISS.

In [3]:
semantic = SemanticRetriever()

faiss_path = INDICES_DIR / "faiss_index"
if faiss_path.exists():
    semantic.load(str(faiss_path))
    print("Semantic index loaded from disk.")
else:
    semantic.build_index(corpus)
    semantic.save(str(faiss_path))
    print("Semantic index built and saved.")

results = semantic.search("vitamin C serum", top_k=5)
print("\nTest query: 'vitamin C serum'")
for i, r in enumerate(results):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n")
results2 = semantic.search("something to protect from sun damage", top_k=5)
print("\nTest query: 'something to protect from sun damage'")
for i, r in enumerate(results2):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n")
results3 = semantic.search("face brightening serum with antioxidants", top_k=5)
print("\nTest query: 'face brightening serum with antioxidants'")
for i, r in enumerate(results3):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2671.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic index loaded from disk.

Test query: 'vitamin C serum'
1. PURE VITAMIN C SERUM (score: 0.9034)
2. Liz K Super First C Serum Pure Vitamin C 13% (score: 0.7300)
3. Vitamin C Serum for Face - C Booster With Hyaluronic Acid and Vitamin E - Anti Wrinkle - Anti Aging - Skin Repair For Age Spots and Sun Damage - For Men and Women (score: 0.7153)
4. Charlotte Elizabeth Organics Advanced Illuminating Vitamin C Serum (score: 0.7137)
5. Collagen SPA Enriching Serum With Vitamin C 1.7 fl oz (score: 0.7114)



Test query: 'something to protect from sun damage'
1. UV Capture PLUS Pure Mild Sun Cream Sun Block Sunscreen SPF PA +++ + Moistful Essential Sun Block Cream With Skin Relieving Protection Barrier (score: 0.6226)
2. SUNPLUS - Laguna - Everyday Inspired SPF40 (score: 0.5780)
3. Svr Sun Secure Solar Water Spf50 + 200ml (score: 0.5733)
4. Surface Sheer Touch Spray Sunscreen - Reef Safe, Ultra-Light & Clean Feeling, Broad Spectrum UVA/UVB Protection, Cruelty & Paraben Free, Water Resista

**Observations on semantic retriever results:**

- `"vitamin C serum"`: Semantic search returns relevant results (all are vitamin C serums) but ranks them differently than BM25. "PURE VITAMIN C SERUM" scores highest (0.90) despite having a short title because the embedding captures meaning rather than rewarding keyword repetition. BM25 favoured longer titles that repeated the phrase multiple times.
- `"something to protect from sun damage"`: None of these results contain the exact words "protect from sun damage" yet the model correctly retrieves sunscreen/sun block products (UV Capture PLUS, SUNPLUS Laguna SPF40, Svr Sun Secure SPF50) by understanding the *intent* behind the query. BM25 would struggle here because the query terms don't match product vocabulary (e.g., "SPF", "sun block", "sunscreen"). However, result #5 (Buddha Balm lip balm) is a false positive. The model associated "sun" broadly without distinguishing lip balm from sun protection.
- `"face brightening serum with antioxidants"`: Semantic search retrieves a diverse mix of brightening serums and antioxidant products (Vitamin C Serum with L'Ascorbic Acid, Enhanced Face Lightening Vitamin C Gel, Super Antioxidant Concentrate Serum, Mouj Lumiere Brightening Serum). This demonstrates the model's ability to connect related concepts that don't share exact keywords (e.g., linking vitamin C to "antioxidant" and "brightening").
- Scores range from ~0.57 to ~0.90 (cosine similarity). Exact keyword matches score highest. Intent-based queries score lower but still retrieve relevant products. This is expected because the model has to "bridge" a larger semantic gap for paraphrased queries.
- `Limitation:` The model can retrieve false positives when query terms have broad associations. For example, a lip balm appearing for a "sun damage" query because both relate to "sun." This suggests the 384-dimensional embedding space sometimes groups loosely related concepts too close together.

## 3. Hybrid Retriever

Combining BM25 and semantic scores with weighted linear combination. The hybrid approach min-max normalizes both score sets to [0, 1], then combines them: `score = bm25_weight * bm25_score + (1 - bm25_weight) * semantic_score`.

In [4]:
hybrid = HybridRetriever(bm25, semantic)

# Test with same queries used in BM25 and semantic sections
print("Test query: 'vitamin C serum' (weight=0.5)")
results = hybrid.search("vitamin C serum", top_k=5, bm25_weight=0.5)
for i, r in enumerate(results):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n")
print("\nTest query: 'something to protect from sun damage' (weight=0.5)")
results2 = hybrid.search("something to protect from sun damage", top_k=5, bm25_weight=0.5)
for i, r in enumerate(results2):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n")
print("\nTest query: 'gentle cleanser for sensitive skin' (weight=0.3, favor semantic)")
results3 = hybrid.search("gentle cleanser for sensitive skin", top_k=5, bm25_weight=0.3)
for i, r in enumerate(results3):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

Test query: 'vitamin C serum' (weight=0.5)
1. PURE VITAMIN C SERUM (score: 0.8363)
2. Organic Vitamin C Serum for Face-Professional Strength-Organic Vitamin C Serum-Hyaluronic Acid-Vitamin E-Naturally Derived 20% Vitamin C (2 Ounce) (score: 0.5000)
3. PREMIUM Vitamin C Serum For Face and Eyes with Hyaluronic Acid Serum - Anti Ageing & Anti Wrinkle Serum - This Vitamin C Serum Will Plump, Hydrate & Brighten, Anti Wrinkle, Anti Aging, Fades Age Spot (score: 0.4902)
4. 24K Vitamin C Serum for Face 2 PACK, Facial Serum with Vitamin C and Hyaluronic Acid, Facial Moisturizer 2 Pack Vitamin C Serum (score: 0.4378)
5. Springs Organic Vitamin C Serum For Your Face - Vitamin C Serum|Organic Vitamin C + Vitamin E + Aloe Vera + Hyaluronic Acid Serum- Effective Strength 20% Vitamin C Serum with Vegan Hyaluronic Acid Leaves Your Skin Radiant & More Beautiful By Neutralizing Free Radicals. This Anti Aging Antiwrinkles Serum Will Give You The Results in few days or we refund You With No-Question-Asked

**Observations on hybrid retriever results:**

- `"vitamin C serum" (weight=0.5)`: The hybrid reranks results compared to both individual methods. "PURE VITAMIN C SERUM" stays at #1 (score 0.84) because it scores well on both keyword match and semantic similarity. The BM25-dominant product (Organic Vitamin C Serum) moves to #2 (score 0.50). It scored highest in BM25 from keyword repetition but its lower semantic score pulls the combined score down. The remaining results follow BM25's order which suggests that when semantic scores are similar, keyword overlap becomes the tiebreaker.
- `"something to protect from sun damage" (weight=0.5)`: The top two products (Vertra Clear SPF 28 Face Stick and UV Capture PLUS) both score 0.50, meaning they scored well across both methods. Unlike BM25 alone, the hybrid avoids matching irrelevant products on the word "something." KOSE Sekkisei SPF50+ at #3 (0.38) and SUNPLUS SPF40 at #4 (0.28) round out the results with genuinely relevant sun protection products.
- `"gentle cleanser for sensitive skin" (weight=0.3, favoring semantic)`: Lowering the BM25 weight to 0.3 lets semantic similarity dominate. Medicalia Gentle Cleanser scores 0.98, well above the rest because it scores highly in both methods (title is a near-exact match for the query *and* the embedding aligns). The gap between #2 (0.73) and #3 (0.32) is large, indicating the top two are clearly relevant while the remaining results are borderline.
- Score range is [0, 1] as expected from min-max normalization. The weight parameter gives a tunable knob, useful in the app where users can adjust based on whether their query is keyword-heavy or intent-heavy.

## 4. Qualitative Evaluation

Running all three retrieval methods on our 21-query evaluation set and comparing results side-by-side.

In [5]:
queries_df = pd.read_csv(DATA_PROCESSED / "ground_truth.csv")

eval_results = []
for _, row in queries_df.iterrows():
    query = row["query"]
    bm25_res = bm25.search(query, top_k=5)
    sem_res = semantic.search(query, top_k=5)
    hyb_res = hybrid.search(query, top_k=5, bm25_weight=0.5)

    eval_results.append({
        "query_id": row["query_id"],
        "query": query,
        "difficulty": row["difficulty"],
        "bm25_top3": " | ".join(r["title"][:50] for r in bm25_res[:3]),
        "semantic_top3": " | ".join(r["title"][:50] for r in sem_res[:3]),
        "hybrid_top3": " | ".join(r["title"][:50] for r in hyb_res[:3]),
    })

eval_df = pd.DataFrame(eval_results)
print(f"Evaluated {len(eval_df)} queries across 3 methods")

diff_colors = {"easy": "background-color: #e8f5e9; color: black", "medium": "background-color: #fff3e0; color: black", "hard": "background-color: #ffebee; color: black"}

(
    eval_df.style
    .map_index(lambda _: "text-align: left", axis=1)
    .set_properties(**{"text-align": "left", "vertical-align": "top", "color": "black"})
    .apply(lambda row: [diff_colors.get(row["difficulty"], "")] * len(row), axis=1)
)

Evaluated 21 queries across 3 methods


,query_id,query,difficulty,bm25_top3,semantic_top3,hybrid_top3
0,1,vitamin C serum,easy,"Organic Vitamin C Serum for Face-Professional Stre | PREMIUM Vitamin C Serum For Face and Eyes with Hya | 24K Vitamin C Serum for Face 2 PACK, Facial Serum",PURE VITAMIN C SERUM | Liz K Super First C Serum Pure Vitamin C 13% | Vitamin C Serum for Face - C Booster With Hyaluron,PURE VITAMIN C SERUM | Organic Vitamin C Serum for Face-Professional Stre | PREMIUM Vitamin C Serum For Face and Eyes with Hya
1,2,coconut oil shampoo,easy,FREY Natural Sulfate Free Shampoo - Natural Shampo | Lauat Herbal Hair Shampoo - 8.45 fl oz bottle of L | J.R.Liggett's Old Fashioned Bar Shampoo Coconut &,"Nyle Anti Dandruf Shampoo 180 | Shampoo - Jojoba Oil, Coconut Oil - Sulfate Free – | Cocabi Natural Shampoo Coconut & Abyssinian Oils D","FREY Natural Sulfate Free Shampoo - Natural Shampo | Shampoo - Jojoba Oil, Coconut Oil - Sulfate Free – | Nyle Anti Dandruf Shampoo 180"
2,3,charcoal face mask,easy,"Blackhead Remover - Activated Charcoal Face Mask - | Blackhead Remover Mask, Beyond Black Mask, Charcoa | Black MasK Peel Off Mask Blackhead Remover Mask Ch",Cocovit - Organic Activated Coconut Charcoal Face | Bamboo Charcoal Deep Clean Mask Remove Peel off Fa | 5 Days Black Charcoal Facial Mask for Skin Elastic,"Blackhead Remover - Activated Charcoal Face Mask - | Cocovit - Organic Activated Coconut Charcoal Face | Blackhead Remover Mask, Beyond Black Mask, Charcoa"
3,4,retinol cream,easy,Danielle Laroche DL Retinol Vitamin A and E Firmin | Hotmir Retinol Moisturizer Cream for Face and Neck | Khali Beauty Retinol Moisturizer Gel Cream for Fac,Spa Ultimate Retinol Anti Aging Night Cream 1.69 F | Skin Effects by Dr. Jeffrey Dover EYE REPAIR SERUM | Retinol Plus Anti Aging Day cream with Retinol and,Spa Ultimate Retinol Anti Aging Night Cream 1.69 F | Danielle Laroche DL Retinol Vitamin A and E Firmin | Hotmir Retinol Moisturizer Cream for Face and Neck
4,5,sunscreen SPF 50,easy,"Cerave Sunscreen Bundle SPF 50 | Contains Mineral | Anti-aging sunscreen SPF 50+ Duolys, 50 ml, Acm | Banana Boat Sunscreen for Men Triple Defense Sunsc","Anti-aging sunscreen SPF 50+ Duolys, 50 ml, Acm | 12-hour full coverage foundation broad spectrum SP | Perfectly Plain Collection Sunscreen with Spf30 (5","Anti-aging sunscreen SPF 50+ Duolys, 50 ml, Acm | Cerave Sunscreen Bundle SPF 50 | Contains Mineral | 12-hour full coverage foundation broad spectrum SP"
5,6,tea tree face wash,easy,"Tea Tree Face Wash 16 fl.oz. Seed + Posy, Clarifyi | VITAMIN C FACE WASH GEL with Amino Acid Complex, H | PURE Tea Tree Oil Shampoo & Conditioner Set, 26.5","Tea Tree Face Wash 16 fl.oz. Seed + Posy, Clarifyi | Way Of Will Tea Tree Body Wash, Moisturizing Body | TISSERAND Tea Tree & Aloe Foaming Face Wash, 150 C","Tea Tree Face Wash 16 fl.oz. Seed + Posy, Clarifyi | Way Of Will Tea Tree Body Wash, Moisturizing Body | TISSERAND Tea Tree & Aloe Foaming Face Wash, 150 C"
6,7,argan oil hair mask,easy,"Spa Life Argan Black Mask - Blackhead Remover - Ac | Virgin Moroccan Argan Oil. Pure, Certified Organic | L'emarie Nutritive Repair Hair Mask Deep Condition","One 'n Only Hair Mask with Argan Oil, Revitalizes | Argan Oil Hair Mask Deep Conditioner - Sulfate Fre | Argan Oil Hair Mask for Dry Damaged Hair - Keratin","One 'n Only Hair Mask with Argan Oil, Revitalizes | Hair Strengthening Deep Conditioner Mask - Thick C | Spa Life Argan Black Mask - Blackhead Remover - Ac"
7,8,something to reduce acne scars,medium,"Lycopene Skin Care – Safe Fruit Acids Serum, Brigh | Acne Scar Removal Cream Stretch Marks Face Skin Cr | Natural Acne Scar Removal Cream for Acne Scars, St","Natural Acne Scar Removal Cream for Acne Scars, St | Acne Scar Reducing Cream, All Natural w/Tomato,Cuc | New York Laboratories Acne Scar Removal Cream, Hel","Natural Acne Scar Removal Cream for Acne Scars, St | Acne Scar Removal Cream Stretch Marks Face Skin Cr | Acne Scar Reducing Cream, All Natural w/Tomato,Cuc"
8,9,gift set that smells like van

In [7]:
# Side-by-side comparison for 5 selected queries (mix of easy, medium, hard)
sample_queries = [
    "vitamin C serum",                                  # easy
    "sunscreen SPF 50",                                 # easy
    "something to reduce acne scars",                   # medium
    "gentle cleanser for sensitive skin",               # medium
    "what helps with sun damage on fair skin"           # hard
]

rows = []
for query in sample_queries:
    difficulty = queries_df[queries_df["query"] == query]["difficulty"].values[0]
    bm25_res = bm25.search(query, top_k=5)
    sem_res = semantic.search(query, top_k=5)
    for i in range(5):
        rows.append({
            "Query": query,
            "Difficulty": difficulty,
            "Rank": i + 1,
            "BM25 Result": bm25_res[i]["title"] if i < len(bm25_res) else "N/A",
            "BM25 Score": f"{bm25_res[i]['score']:.2f}" if i < len(bm25_res) else "",
            "Semantic Result": sem_res[i]["title"] if i < len(sem_res) else "N/A",
            "Semantic Score": f"{sem_res[i]['score']:.2f}" if i < len(sem_res) else "",
        })

comparison_df = pd.DataFrame(rows)

diff_colors = {"easy": "background-color: #e8f5e9; color: black", "medium": "background-color: #fff3e0; color: black", "hard": "background-color: #ffebee; color: black"}

with pd.option_context("display.max_colwidth", None):
    display(
        comparison_df.style
        .map_index(lambda _: "text-align: left", axis=1)
        .set_properties(**{"text-align": "left", "vertical-align": "top", "color": "black"})
        .apply(lambda row: [diff_colors.get(row["Difficulty"], "")] * len(row), axis=1)
        .hide(axis="index")
    )

Query,Difficulty,Rank,BM25 Result,BM25 Score,Semantic Result,Semantic Score
vitamin C serum,easy,1,Organic Vitamin C Serum for Face-Professional Strength-Organic Vitamin C Serum-Hyaluronic Acid-Vitamin E-Naturally Derived 20% Vitamin C (2 Ounce),21.65,PURE VITAMIN C SERUM,0.90
vitamin C serum,easy,2,"PREMIUM Vitamin C Serum For Face and Eyes with Hyaluronic Acid Serum - Anti Ageing & Anti Wrinkle Serum - This Vitamin C Serum Will Plump, Hydrate & Brighten, Anti Wrinkle, Anti Aging, Fades Age Spot",21.61,Liz K Super First C Serum Pure Vitamin C 13%,0.73
vitamin C serum,easy,3,"24K Vitamin C Serum for Face 2 PACK, Facial Serum with Vitamin C and Hyaluronic Acid, Facial Moisturizer 2 Pack Vitamin C Serum",21.41,Vitamin C Serum for Face - C Booster With Hyaluronic Acid and Vitamin E - Anti Wrinkle - Anti Aging - Skin Repair For Age Spots and Sun Damage - For Men and Women,0.72
vitamin C serum,easy,4,Springs Organic Vitamin C Serum For Your Face - Vitamin C Serum|Organic Vitamin C + Vitamin E + Aloe Vera + Hyaluronic Acid Serum- Effective Strength 20% Vitamin C Serum with Vegan Hyaluronic Acid Leaves Your Skin Radiant & More Beautiful By Neutralizing Free Radicals. This Anti Aging Antiwrinkles Serum Will Give You The Results in few days or we refund You With No-Question-Asked. Your!,21.36,Charlotte Elizabeth Organics Advanced Illuminating Vitamin C Serum,0.71
vitamin C serum,easy,5,Mererke_Pretty Vitamin C Serum for Face: Vitamin C 20 Facial and Under Eye Serum with Hyaluronic Acid - Wrinkle Remover Serum to Even and Tone Skin - Anti Aging and Brightening Skin Care Serums - 1 F,21.27,Collagen SPA Enriching Serum With Vitamin C 1.7 fl oz,0.71
sunscreen SPF 50,easy,1,"Cerave Sunscreen Bundle SPF 50 | Contains Mineral Sunscreen for Face SPF 50, 2.5 Ounce, and Mineral Body Sunscreen SPF 50, 5 Ounce 1 ea",26.10,"Anti-aging sunscreen SPF 50+ Duolys, 50 ml, Acm",0.79
sunscreen SPF 50,easy,2,"Anti-aging sunscreen SPF 50+ Duolys, 50 ml, Acm",21.91,"12-hour full coverage foundation broad spectrum SPF 15 sunscreen, deep sand 1.7 fl oz (50 ml)",0.79
sunscreen SPF 50,easy,3,"Banana Boat Sunscreen for Men Triple Defense Sunscreen Spray, SPF 50, 6 Oz (Pack of 3)",21.72,Perfectly Plain Collection Sunscreen with Spf30 (50),0.78
sunscreen SPF 50,easy,4,"Coppertone ClearlySheer Faces Sunscreen SPF 50 Lotion, 2 Ounce",21.61,Mary Kay CC Cream Sunscreen Broad Spectrum SPF 15 1fl. oz / 29 mL - Deep (2-Pack),0.77
sunscreen SPF 50,easy,5,Heliocare Ultra SPF 50+ Gel Facial Sunscreen 50ml,21.61,Elaine Gregg Advanced Protection SPF 30 3.3 oz,0.77


### Discussion & Comparison: BM25 vs Semantic vs Hybrid

**Per-Query Analysis (5 selected queries):**

1. **"vitamin C serum" [easy]** - Both methods retrieve relevant products. BM25 ranks the longest titles first (more keyword repetitions = higher term frequency), while semantic ranks "PURE VITAMIN C SERUM" #1 for direct meaning alignment. Hybrid combines both signals effectively. **Winner: Tie** as easy keyword queries work well for both.

2. **"sunscreen SPF 50" [easy]** - BM25 is more precise: all top-5 contain exactly "SPF 50" or "SPF 50+" (Cerave, Banana Boat, Coppertone, Heliocare). Semantic retrieves an SPF 50+ product at #1 but also returns a "12-hour full coverage foundation" (#2) and an SPF 30 product (#5) because it understands "sunscreen" conceptually but cannot enforce numeric constraints. **Winner: BM25** as exact specification matching favors keyword search.

3. **"something to reduce acne scars" [medium]** - BM25 returns mostly relevant products (Acne Scar Removal Cream, Natural Acne Scar Removal Cream) but is distracted by tangential matches: Lycopene Skin Care at #1 is about reducing fine lines and fighting acne broadly, not specifically scar treatment, and tarte Mini Timeless Smoothing Primer at #4 is a cosmetic primer, not a scar treatment. Semantic returns a more focused set of scar-treatment products (Natural Acne Scar Removal Cream, Acne Scar Reducing Cream, New York Labs Acne Scar Removal Cream, MicroDermabrasion Cloth). **Winner: Semantic** as it handles vague phrasing and retrieves more focused results.

4. **"gentle cleanser for sensitive skin" [medium]** - BM25 finds products explicitly labeled "gentle" and "sensitive skin" (Natural Face Wash for Sensitive Skin, Medicalia Gentle Cleanser, PHL Naturals Sensitive Skin). Semantic surfaces mostly relevant products but includes SELF/ish Men's Face Scrub (#4) (an *exfoliating* scrub), which is the opposite of "gentle." **Winner: BM25** as keyword precision matters when the query uses specific product vocabulary.

5. **"what helps with sun damage on fair skin" [hard]** - Both struggle significantly. BM25 returns largely irrelevant products such as Flawless Finish Foundation at #1 matched on "skin" and "fair," Josie Maran Self Tanning Cream at #2 matched on "skin" and tanning-related terms, and COL-LAB Sun Obsession Sculpting Bronzer at #5 is a cosmetic bronzer with no sun-repair function. Semantic does better thematically, Gold Bond Dark Spot Minimizing Cream (#1, 0.65) and Ban The Sun After Sun Soothing Gel (#2, 0.64) are at least related to sun damage recovery, but scores are low across the board (0.62-0.65) which reflects the model's difficulty bridging a conversational, multi-concept query to product titles. **Winner: Semantic** as it captures the sun-damage theme better, but neither method handles the combined intent of "sun damage" + "fair skin" + "what helps" well.

---

**Where BM25 Fails but Semantic Succeeds:**
- **Intent-based queries** (e.g., "something to protect from sun damage"): BM25 cannot bridge the vocabulary gap between "protect" and "SPF/sunscreen." Semantic maps intent correctly.
- **Paraphrased queries** (e.g., "product to make hair less frizzy"): BM25 needs exact overlap; semantic connects "less frizzy" to anti-frizz products.
- **Conversational filler** (e.g., "something to keep skin hydrated all day"): The word "something" is noise for BM25 but invisible to the embedding model.

**Where Semantic Fails:**
- **Exact specifications**: Retrieves SPF 30 and non-sunscreen products for an "SPF 50" query. Embeddings treat numbers as semantically similar and cannot enforce exact numeric matches.
- **Product-type confusion**: An exfoliating scrub appears for a "gentle cleanser" query. The model associates "cleanser" broadly without distinguishing gentle from abrasive formulations.
- **Vague multi-concept queries**: For "what helps with sun damage on fair skin," semantic scores are uniformly low (0.62-0.65), showing the model cannot confidently match a conversational, multi-concept query to product-level descriptions.

---

**Performance by Query Difficulty:**

| Difficulty | BM25 | Semantic | Hybrid |
|------------|------|----------|--------|
| **Easy** (exact keywords) | Strong - direct keyword match | Strong - but may include wrong specs | Strong - combines both signals |
| **Medium** (paraphrased/intent) | Mixed - fails on vague phrasing | Better - captures intent without keyword overlap | Best - semantic handles intent, BM25 boosts exact matches |
| **Hard** (multi-concept/conversational) | Weak - false positives from individual word matches | Moderate - captures theme but scores are low and results lack precision | Moderate - inherits limitations of both |

**Key Takeaways:**
1. **BM25 excels at precision** for keyword-specific queries but fails when users paraphrase or describe needs conversationally.
2. **Semantic search excels at recall** for intent-based queries but cannot enforce exact specifications or structured constraints like price.
3. **Hybrid retrieval provides the best balance**, surfacing products that score well across both dimensions.
4. **Neither method handles multi-concept conversational queries well** (e.g., combining sun damage + fair skin + remedies). This is the gap a RAG pipeline could fill. An LLM can reason over retrieved candidates and filter on structured attributes like price and ratings.